# 面试问题：Causal LM 的 label shift、loss、perplexity 和 bits-per-byte 怎样正确计算？

**一句话回答**：位置 `t` 的 logits 预测下一个 token `x_{t+1}`，因此 inputs/labels 要错位一格；稳定交叉熵只在有效 target token 上聚合，PPL 是全局平均 NLL 的指数，不能先算每条 PPL 再平均。不同 tokenizer 的 token 粒度不同，比较时用总字节归一化的 BPB 更公平。

本 Notebook 用 PyTorch 基础张量手写 log-softmax、gather loss、mask、packing 边界、滑窗评测和 BPB，并用中文注释标出最常见的 off-by-one 与分母错误。


In [ ]:
import math
import numpy as np
import torch

# 固定输入规模，使每个数值断言都可复现。
SEED140=14001; torch.manual_seed(SEED140)
assert SEED140==14001
assert math.isclose(math.exp(math.log(5)),5)
assert torch.isfinite(torch.randn(2)).all()


## 1. Shift 后才形成监督对

序列 `[BOS, A, B, EOS]` 的模型输入通常是前三个 token，标签是后三个 token。首个 BOS 没有前文预测目标，最后输入位置预测 EOS。框架有的内部 shift、有的要求外部 shift；重复 shift 会把任务错成预测下下个 token。


In [ ]:
def shift140(tokens):
    # 每个 input 位置只监督紧邻的下一个 token。
    t=torch.as_tensor(tokens); return t[:-1],t[1:]
inp140,lab140=shift140([1,5,6,2])
assert torch.equal(inp140,torch.tensor([1,5,6]))
assert torch.equal(lab140,torch.tensor([5,6,2]))
assert len(inp140)==len(lab140)==3


## 2. Attention causal mask 与 label shift 是两份不同合同

causal mask 禁止隐藏状态读取未来 token；label shift 决定该隐藏状态监督哪个 target。只做 shift 不做 causal mask 会训练时偷看答案；只做 mask 不 shift 则监督当前输入 token。padding 与文档边界还需叠加额外 mask。


In [ ]:
def causal_mask140(n):
    # True 表示 query 行允许读取对应 key 列。
    return torch.arange(n)[:,None]>=torch.arange(n)[None,:]
cm140=causal_mask140(4)
assert cm140.shape==(4,4)
assert bool(cm140[3,0]) and not bool(cm140[0,3])
assert int(cm140.sum())==10


## 3. 稳定交叉熵用 log-sum-exp 与 target gather

`logp=logits-logsumexp(logits)`，再取 label 对应位置的负 log-prob。mask 后的分母是有效 target 数，而不是 batch×max_len。这里不调用 `cross_entropy`，便于看到数值稳定和 ignore 语义。


In [ ]:
def token_nll140(logits,labels):
    # 先做稳定 log-softmax，再按标签索引抽取目标概率。
    logp=logits-torch.logsumexp(logits,dim=-1,keepdim=True)
    return -logp.gather(-1,labels[...,None]).squeeze(-1)
logits140=torch.tensor([[1000.,1001.,999.],[1.,0.,3.],[0.,0.,0.]])
nll140=token_nll140(logits140,torch.tensor([1,2,0]))
assert torch.isfinite(nll140).all()
assert nll140[1]<nll140[2]
assert math.isclose(float(nll140[2]),math.log(3),rel_tol=1e-6)


## 4. PPL 是 token 加权总 NLL 的指数

对不同长度样本，先汇总 `ΣNLL/Σvalid_tokens` 再取 exp；直接平均各样本 PPL 会让短样本权重过大，且 Jensen 不等式使结果不同。报告 PPL 时同时给 tokenizer、数据版本、窗口和是否预测 EOS。


In [ ]:
def corpus_ppl140(nll_rows,masks):
    # 全语料只除一次有效 token 总数。
    total=sum(float((n*m).sum()) for n,m in zip(nll_rows,masks)); count=sum(float(m.sum()) for m in masks)
    return math.exp(total/count)
rows140=[torch.tensor([.1,.2,0.]),torch.tensor([2.,0.,0.])]; masks140=[torch.tensor([1.,1.,0.]),torch.tensor([1.,0.,0.])]
ppl140=corpus_ppl140(rows140,masks140); wrong140=np.mean([math.exp(.15),math.exp(2.)])
assert math.isclose(ppl140,math.exp(2.3/3),rel_tol=1e-6)
assert not math.isclose(ppl140,wrong140)
assert ppl140>1


## 5. 跨 tokenizer 用 byte-normalized 指标

token PPL 会随切词粒度变化：更细 token 往往单 token 更容易，却需要更多步。`BPB=total_nll_nats/(bytes·ln2)` 表示编码每个原始字节需要多少 bit，在相同文本字节上更可比；Unicode 正规化和字节口径仍要固定。


In [ ]:
def bpb140(total_nll_nats,num_bytes):
    # nats 除以 ln(2) 转 bit，再按原始 UTF-8 字节归一化。
    return total_nll_nats/(num_bytes*math.log(2))
text140="你好A"; bytes140=len(text140.encode("utf-8"))
assert bytes140==7
assert math.isclose(bpb140(7*math.log(2),bytes140),1.0)
assert bpb140(3.5,bytes140)>0


## 6. Packing 后要阻断跨文档 attention 与 loss

把多个短文档拼入一行可减少 padding，但若不加 segment mask，文档 B 会读取 A；若不屏蔽跨界 target，A 的 EOS/末 token 会被要求预测 B 的 BOS。是否允许跨样本共享上下文是训练配方的一部分，不应由 packer 默认决定。


In [ ]:
seg140=torch.tensor([0,0,0,1,1]); n140=len(seg140)
# 同时满足因果顺序和 segment 相同，才允许 attention。
packed_attn140=causal_mask140(n140)&(seg140[:,None]==seg140[None,:])
next_same140=seg140[:-1]==seg140[1:]
assert not bool(packed_attn140[3,2])
assert bool(packed_attn140[4,3])
assert torch.equal(next_same140,torch.tensor([True,True,False,True]))


## 7. 长文本滑窗评测不能重复计分重叠 token

窗口有 overlap 是为了给新 token 提供上下文；每个 target 只在第一次拥有足够上下文的窗口计分。下面生成窗口的新增评分区间，并证明除第一个无前文 token 外覆盖恰好一次。


In [ ]:
def score_ranges140(length,window,stride):
    # overlap 只提供上下文，score_start 之后才计入本窗口损失。
    end=min(window,length); ranges=[(0,1,end)]
    while end<length:
        new_end=min(length,end+stride); start=max(0,new_end-window)
        ranges.append((start,end,new_end)); end=new_end
    return ranges
ranges140=score_ranges140(10,6,3); coverage140=np.zeros(10,int)
for _,s,e in ranges140: coverage140[s:e]+=1
assert np.all(coverage140[1:]==1)
assert coverage140[0]==0
assert ranges140[-1][2]==10


## 8. PPL 还需按 slice、置信与泄漏解释

同一平均 NLL 可能隐藏代码、长文本或低资源语言退化。同步报告 token accuracy、top-k、长度/域 slice、校准和 byte 口径；训练数据污染会让 PPL 虚低。线上生成质量还受 decoding 和 prompt 模板影响，不能由 teacher-forced PPL 单独代表。


In [ ]:
probs140=torch.softmax(logits140,dim=-1); pred140=probs140.argmax(-1); labels140=torch.tensor([1,2,0])
# accuracy 只看 argmax，NLL 还反映概率置信度。
acc140=float((pred140==labels140).float().mean()); mean_nll140=float(nll140.mean())
assert acc140==1.0
assert mean_nll140>0
assert not math.isclose(acc140,mean_nll140)


## 面试总结

回答顺序建议是：**明确 BOS/EOS → logits 与 labels 错一位 → causal/segment/padding mask 分层 → 手写稳定 log-softmax + gather → 全局 `ΣNLL/Σtoken` 后 exp → 跨 tokenizer 用 BPB → packing 阻断跨文档 → 滑窗 overlap 只提供上下文不重复计分 → 按域/长度和污染审计解释指标**。PPL 是压缩式概率指标，不等于开放生成的完整质量。

延伸阅读：[GPT-3](https://arxiv.org/abs/2005.14165)、[Chinchilla](https://arxiv.org/abs/2203.15556)、[The Pile](https://arxiv.org/abs/2101.00027)。
